# 01 — Нормализация текста-шаблона
## Цель
Загрузить FB2-файл, извлечь весь печатный текст в порядке документа, нормализовать и подготовить единую координатную строку для дальнейшего сопоставления с аудио и страницами.

## Структура FB2
- Обрабатываются все `<body>` FB2 в исходном порядке.
- Включаются абзацы, подзаголовки, стихотворные строки и текст авторства из всех секций, включая приложения и комментарии.
- Метаданные из `<description>` и бинарные данные не входят в печатный текст и не извлекаются.

## Выход
- `normalized_text` — строка всего текста в нижнем регистре
- `char_index_to_source` — dict: позиция символа → путь `body`/секции
- `section_boundaries` — границы последовательных секций для диагностики

In [1]:
import pickle, re, os
from pathlib import Path
from lxml import etree

# Корень проекта — для переносимости определяем относительно этого файла
PROJECT_ROOT = Path(os.environ.get(
    "SPARK_ROOT",
    "/home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit"
))

In [2]:
# Пути
DATA_DIR = PROJECT_ROOT / "data/besy"
OUTPUT_DIR = PROJECT_ROOT / "outputs/besy/run_01"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FB2_PATH = DATA_DIR / "text/besy.fb2"
print(f"FB2: {FB2_PATH}")
print(f"Exists: {FB2_PATH.exists()}")

FB2: /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/data/besy/text/besy.fb2
Exists: True


In [3]:
tree = etree.parse(str(FB2_PATH))
root = tree.getroot()
NS = "http://www.gribuser.ru/xml/fictionbook/2.0"

def tag(name):
    return f"{{{NS}}}{name}"

In [4]:
def extract_paragraphs(section_element):
    """Извлечь все <p> внутри секции (прямые и вложенные)."""
    return section_element.findall(f".//{tag('p')}")

def paragraph_text(paragraph_element):
    """Вернуть полный текст абзаца, включая текст внутри выделений и ссылок."""
    return ''.join(paragraph_element.itertext()).strip()

def get_title(section_element):
    """Извлечь заголовок секции."""
    title_el = section_element.find(tag('title'))
    if title_el is not None:
        title_p = title_el.find(tag('p'))
        if title_p is not None:
            return paragraph_text(title_p)
    return ""

In [5]:
bodies = root.findall(tag('body'))
assert bodies, 'В FB2 не найдено ни одного <body>.'

print(f'Найдено body: {len(bodies)}')
for body_index, body in enumerate(bodies):
    print(f'  Body {body_index}: {len(list(body.iter(tag("p"))))} абзацев')

Найдено body: 3
  Body 0: 5883 абзацев
  Body 1: 1318 абзацев
  Body 2: 642 абзацев


In [6]:
TEXT_BLOCK_TAGS = {tag('p'), tag('subtitle'), tag('text-author'), tag('v')}


def source_label(element, body_index):
    """Построить путь body/секций для диагностики позиции текста."""
    titles = []
    parent = element.getparent()
    while parent is not None and parent.tag != tag('body'):
        if parent.tag == tag('section'):
            title = get_title(parent)
            if title:
                titles.append(title)
        parent = parent.getparent()
    titles.reverse()
    return f"Body {body_index}" + (f" / {' / '.join(titles)}" if titles else '')


def extract_body_blocks(body, body_index):
    """Вернуть все печатные текстовые блоки body в порядке XML-документа."""
    blocks = []
    for element in body.iter():
        if element.tag in TEXT_BLOCK_TAGS:
            text = paragraph_text(element)
            if text:
                blocks.append((text, source_label(element, body_index)))
    return blocks

In [7]:
# Вспомогательные функции определены выше; отдельные секции не фильтруются.

In [8]:
all_paragraphs = []
for body_index, body in enumerate(bodies):
    blocks = extract_body_blocks(body, body_index)
    all_paragraphs.extend(blocks)
    print(f'Body {body_index}: добавлено текстовых блоков {len(blocks)}')

assert all_paragraphs, 'В body не найдено печатных текстовых блоков.'
print(f'Всего текстовых блоков: {len(all_paragraphs)}')

Body 0: добавлено текстовых блоков 6115
Body 1: добавлено текстовых блоков 1318
Body 2: добавлено текстовых блоков 692
Всего текстовых блоков: 8125


In [9]:
print('В корпус включены все body и секции без фильтрации по заголовкам.')

В корпус включены все body и секции без фильтрации по заголовкам.


In [10]:
def normalize_text(text):
    """Нормализовать текст параграфа для сопоставления."""
    text = text.lower()
    text = re.sub(r'[^\w\s\-.,!?;:"\'Ѐ-ӿ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Нормализуем параграфы
normalized_paragraphs = []
for raw_text, source in all_paragraphs:
    norm_text = normalize_text(raw_text)
    if norm_text:
        normalized_paragraphs.append((norm_text, source))

# Строим итоговую строку и индекс
char_index_to_source = {}
section_boundaries = []
char_pos = 0
current_section = None
section_start = 0

for i, (norm_text, source) in enumerate(normalized_paragraphs):
    # Границы секций
    if source != current_section:
        if current_section is not None:
            section_boundaries.append((section_start, char_pos - 1, current_section))
        current_section = source
        section_start = char_pos

    # Символы параграфа
    for ch in norm_text:
        char_index_to_source[char_pos] = source
        char_pos += 1

    # Разделитель \n между параграфами (но не после последнего)
    if i < len(normalized_paragraphs) - 1:
        char_index_to_source[char_pos] = source
        char_pos += 1

# Последняя секция
if current_section is not None:
    section_boundaries.append((section_start, char_pos - 1, current_section))

# Итоговая строка
normalized_text = "\n".join(p[0] for p in normalized_paragraphs)

assert len(normalized_text) == char_pos, \
    f"Расхождение: текст={len(normalized_text)}, индекс={char_pos}"

print(f"Длина нормализованного текста: {len(normalized_text):,} символов")
print(f"Количество параграфов: {len(normalized_paragraphs)}")
print(f"Записей в индексе: {len(char_index_to_source):,}")
print(f"Совпадение: текст == индекс ✓")

Длина нормализованного текста: 1,881,785 символов
Количество параграфов: 8123
Записей в индексе: 1,881,785
Совпадение: текст == индекс ✓


In [11]:
# Примеры первых и последних параграфов
print("=== Первые 3 параграфа ===")
for i, (text, source) in enumerate(all_paragraphs[:3]):
    norm = normalize_text(text)
    if norm:
        print(f"[{source}] {norm[:150]}...")

print("\n=== Последние 3 параграфа ===")
for i, (text, source) in enumerate(all_paragraphs[-3:]):
    norm = normalize_text(text)
    if norm:
        print(f"[{source}] {norm[:150]}...")

=== Первые 3 параграфа ===
[Body 0] федор михайлович достоевский...
[Body 0] собрание сочинений в пятнадцати томах...
[Body 0] том 7. бесы...

=== Последние 3 параграфа ===
[Body 2 / 311] с. 661. если соблазните единого от малых сих начальные слова евангельского текста: а кто соблазнит одного из малых сих, верующих в меня, тому лучше бы...
[Body 2 / 312] 312...
[Body 2 / 312] с. 663. эпитимья греч. исполнение исповедавшимся каких-либо благочестивых дел, назначенных ему духовником например, продолжительная молитва, усиленный...


In [12]:
# Статистика по частям
print("=== Распределение текста по частям ===")
for start, end, title in section_boundaries:
    length = end - start + 1
    pct = length / len(normalized_text) * 100
    # Показать фрагмент текста для проверки
    snippet = normalized_text[start:start+80]
    print(f"{title:40s} | {start:>8,}–{end:>8,} | {length:>8,} симв | {pct:5.1f}% | \"{snippet}…\"")

=== Распределение текста по частям ===
Body 0                                   |        0–      78 |       79 симв |   0.0% | "федор михайлович достоевский
собрание сочинений в пятнадцати томах
том 7. бесы
б…"
Body 0 / Бесы                            |       79–     822 |      744 симв |   0.0% | "бесы
хоть убей, следа не видно,
сбились мы, что делать нам?
в поле бес нас водит…"
Body 0 / Бесы / Часть первая             |      823–     835 |       13 симв |   0.0% | "часть первая
глава первая
вместо введения: несколько подробностей из биографии м…"
Body 0 / Бесы / Часть первая / Глава первая |      836–  68,452 |   67,617 симв |   3.6% | "глава первая
вместо введения: несколько подробностей из биографии многочтимого с…"
Body 0 / Бесы / Часть первая / Глава вторая |   68,453– 147,561 |   79,109 симв |   4.2% | "глава вторая
принц гарри , сватовство
i
на земле существовало еще одно лицо, к к…"
Body 0 / Бесы / Часть первая / Глава третья |  147,562– 231,045 |   83,484 симв |   4.4% | "гла

In [13]:
# Сохраняем результаты
outputs = {
    "normalized_text": normalized_text,
    "char_index_to_source": char_index_to_source,
    "section_boundaries": section_boundaries,
    "paragraph_count": len(normalized_paragraphs),
    "source_file": str(FB2_PATH),
}

with open(OUTPUT_DIR / "normalized_text.pkl", "wb") as f:
    pickle.dump(outputs, f)

# Также сохраняем нормализованный текст как TXT для ручного просмотра
with open(OUTPUT_DIR / "normalized_text.txt", "w", encoding="utf-8") as f:
    f.write(normalized_text)

print(f"Сохранено в {OUTPUT_DIR}:")
print(f"  - normalized_text.pkl ({len(normalized_text):,} симв)")
print(f"  - normalized_text.txt (для ручной проверки)")

Сохранено в /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/run_01:
  - normalized_text.pkl (1,881,785 симв)
  - normalized_text.txt (для ручной проверки)


## Проверка

1. Открыть `outputs/besy/run_01/normalized_text.txt` в текстовом редакторе
2. Взять случайный фрагмент текста (например, середину главы)
3. Открыть оригинальный FB2 и найти этот же фрагмент
4. Убедиться, что нормализация не исказила слова и не нарушила порядок
5. Проверить, что приложения и комментарии присутствуют в том месте документа, где они находятся в FB2